# Ćwiczenie z `5_extra.pl.ipynb`: Agent Loop od zera

Przykładowe, niezależne rozwiązanie ćwiczenia z `5_extra.pl.ipynb` (ostatnia komórka tamtego notatnika: "spróbuj zbudować Agent Loop od zera"). Pełny Agent Loop w jednym notatniku, bez odwołań do stanu z pliku źródłowego — ten sam zestaw funkcji (TUI przez `rich`, narzędzie Checklist), napisany samodzielnie jako punkt odniesienia.


In [ ]:
# Importy i inicjalizacja klienta

from rich.console import Console  # sformatowany tekst w terminalu
from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API) z pliku .env
from anthropic import Anthropic  # klient SDK Anthropic
import json  # serializacja wyników narzędzi

load_dotenv(override=True)
anthropic = Anthropic()  # klucz brany z ANTHROPIC_API_KEY w .env


def show(text: str) -> None:  # wypisuje tekst w terminalu, z fallbackiem gdy markup rich jest niepoprawny
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
# Stan checklisty i narzędzia, które Claude będzie wywoływać

checklist: list[str] = []
completed: list[bool] = []


def get_checklist_report() -> str:  # buduje i wypisuje aktualny stan checklisty
    lines = []
    for i, item in enumerate(checklist):
        if completed[i]:
            lines.append(f"[green]☑ {i + 1}. [strike]{item}[/strike][/green]")
        else:
            lines.append(f"☐ {i + 1}. {item}")
    report = "\n".join(lines)
    show(report)
    return report


def create_checklist(descriptions: list[str]) -> str:  # narzędzie: dodaje nowe kroki do checklisty
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()


def mark_complete(index: int, completion_notes: str) -> str:  # narzędzie: oznacza krok jako ukończony
    if not (1 <= index <= len(checklist)):
        return "Błędny indeks checklisty."
    completed[index - 1] = True
    show(f"[dim]{completion_notes}[/dim]")
    return get_checklist_report()

In [ ]:
# Definicje narzędzi w formacie Anthropic (input_schema, nie parameters jak w OpenAI)

create_checklist_json = {
    "name": "create_checklist",
    "description": "Utwórz checklistę kroków na podstawie listy opisów i zwróć jej aktualny stan",
    "input_schema": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Opisy kolejnych kroków checklisty",
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Oznacz krok checklisty (indeks liczony od 1) jako ukończony i zwróć aktualny stan",
    "input_schema": {
        "type": "object",
        "properties": {
            "index": {"type": "integer", "description": "Indeks kroku liczony od 1"},
            "completion_notes": {"type": "string", "description": "Notatka o sposobie ukończenia kroku"},
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False,
    },
}

tools = [create_checklist_json, mark_complete_json]

In [ ]:
# System prompt oraz pętla agentowa

system_message = """
Otrzymujesz problem do rozwiązania. Zanim zaczniesz, ułóż plan jako checklistę narzędziem create_checklist,
a potem wykonuj kolejne kroki, oznaczając każdy jako ukończony narzędziem mark_complete.
Jeśli w treści problemu brakuje jakiejś wielkości, przyjmij rozsądne założenie i zapisz je jako osobny krok checklisty.
Nie zadawaj pytań doprecyzowujących - odpowiedz wynikiem końcowym po wykonaniu wszystkich kroków,
w formacie rich console markup, bez bloków kodu.
"""


def handle_tool_calls(tool_use_blocks: list) -> list[dict]:  # wykonuje wywołania narzędzi i buduje tool_result
    results = []
    for block in tool_use_blocks:
        tool = globals().get(block.name)
        output = tool(**block.input) if tool else f"Nieznane narzędzie: {block.name}"
        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(output),
        })
    return results


def loop(messages: list) -> None:  # pętla agentowa: woła Claude, wykonuje narzędzia, powtarza aż dostaniemy tekst
    response = anthropic.messages.create(
        model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages, tools=tools
    )
    while response.stop_reason == "tool_use":
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]
        results = handle_tool_calls(tool_use_blocks)
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": results})
        response = anthropic.messages.create(
            model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages, tools=tools
        )
    show(next(block.text for block in response.content if block.type == "text"))

In [ ]:
# Uruchomienie: to samo zadanie co w 5_extra.pl.ipynb, inne miasta

checklist, completed = [], []

user_message = """
Pociąg wyjeżdża z Krakowa o 13:00, jadąc z prędkością 90 km/h.
Inny pociąg wyjeżdża z Warszawy o 13:30, jadąc z prędkością 110 km/h w stronę Krakowa.
Kiedy się spotkają?
"""

messages = [{"role": "user", "content": user_message}]
loop(messages)